In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import SimpleITK as sitk
from radiomics import featureextractor


DATA_DIR = "./data"
OUTPUT_CSV = "pyradiomics_features_extraction.csv"

MODALITIES = ["t1gd", "adc", "flair", "t1eg", "t2star", "t2tse", 't1tse']

extractor_mri = featureextractor.RadiomicsFeatureExtractor("params_MRI.yaml")
extractor_adc = featureextractor.RadiomicsFeatureExtractor("params_adc.yaml")


def normalize_mri(image_path, brain_mask_path):
    img_sitk = sitk.ReadImage(image_path)
    mask_sitk = sitk.ReadImage(brain_mask_path)

    data = sitk.GetArrayFromImage(img_sitk).astype(np.float32)
    brain_mask_array = sitk.GetArrayFromImage(mask_sitk)

    bool_mask = brain_mask_array > 0

    mean_brain = data[bool_mask].mean()
    std_brain  = data[bool_mask].std()
    print(std_brain)

    if std_brain == 0:
        std_brain = 1e-6

    data_norm = np.where(bool_mask, ((data - mean_brain) / std_brain) * 100, 0.0)

    norm_sitk = sitk.GetImageFromArray(data_norm)
    norm_sitk.CopyInformation(img_sitk)

    return norm_sitk


all_results = []

patient_folders = [f.path for f in os.scandir(DATA_DIR) if f.is_dir()]

print(f"{len(patient_folders)} patients.\n")

for patient_path in sorted(patient_folders):
    patient_id = os.path.basename(patient_path)

    temporality_folders = [f.path for f in os.scandir(patient_path) if f.is_dir()]

    for temp_path in sorted(temporality_folders):
        time_point = os.path.basename(temp_path)

        print(f"--- {patient_id} | {time_point} ---")


        gtv_files = glob.glob(os.path.join(temp_path, "*gtv*.nii.gz"))
        if not gtv_files:
            print(f" No GTV in {temp_path}.")
            continue
        gtv_path = gtv_files[0]

        brain_mask_files = glob.glob(os.path.join(temp_path, "*brain_mask*.nii.gz"))
        if not brain_mask_files:
            print(f" No brain mask for {patient_id} ({time_point}).")
            continue
        brain_mask_path = brain_mask_files[0]

        for modality in MODALITIES:
            mri_files = glob.glob(os.path.join(temp_path, f"*{modality}*.nii.gz"))

            if not mri_files:
                continue

            mri_path = mri_files[0]
            print(f"  -> Extraction: {modality}...")

            try:
                if modality.lower() == "adc":
                    image_to_extract = sitk.ReadImage(mri_path)
                    result = extractor_adc.execute(image_to_extract, gtv_path)
                else:
                    image_to_extract = normalize_mri(mri_path, brain_mask_path)
                    result = extractor_mri.execute(image_to_extract, gtv_path)

                lumiere_metadata = {
                    "Patient": patient_id,
                    "Time point": time_point,
                    "Image": os.path.basename(mri_path),
                    "Mask": os.path.basename(gtv_path),
                    "Label name": "GTV",
                    "Label": 1,
                    "Sequence": modality,
                }

                full_row = {**lumiere_metadata, **result}
                all_results.append(full_row)

            except Exception as e:
                print(f" Error: {modality} for {patient_id} ({time_point}): {e}")


if all_results:
    df = pd.DataFrame(all_results)

    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\n {len(df)} MRI and '{OUTPUT_CSV}'.")
else:
    print("\n No data.")